# Kitchen Sink Models: Full_pesticides_raw

Trains the full set of model types (Simple LR, Ridge/Lasso/ElasticNet, and tuned XGBoost) **using only** the wide raw pesticide matrix (`pesticide_*_kg` for ~400+ compounds) plus the shared `BASE_COLS`.


In [ ]:
# --- Setup (run first) ---
# 1) Import libraries and sklearn pieces used everywhere below.
# 2) REPO_ROOT / DATA_DIR / RESULTS_DIR so paths work from repo or modeling/.
# 3) TARGET_PATHS, DEFAULT_DROP_COLUMNS, ALPHA_GRID, CV_FOLDS, TARGETS.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style("whitegrid")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "modeling" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "modeling" / "results"

TARGET_PATHS = {
    "CASTHMA": {"train": DATA_DIR / "train_CASTHMA.csv", "test": DATA_DIR / "test_CASTHMA.csv"},
    "COPD": {"train": DATA_DIR / "train_COPD.csv", "test": DATA_DIR / "test_COPD.csv"},
}

DEFAULT_DROP_COLUMNS = {
    "index", "FIPS", "NAME", "state_fips", "CASTHMA", "COPD",
    "CSMOKING", "OBESITY", "DIABETES", "cat3_CASTHMA", "cat3_COPD",
}

ALPHA_GRID = np.logspace(-3, 3, 13)
RANDOM_STATE = 42
CV_FOLDS = 5
N_BOOT = 100   # bootstrap samples for test-metric CIs

# Run for both targets
TARGETS = ["CASTHMA", "COPD"]

In [ ]:
# Embedded into model_selection data cell — four exposure designs + engineer

def engineer_signal_isolation_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per-capita / per-cropland-acre loads, log1p intensities, rural_binary."""

    def safe_divide(num, denom):
        denom_arr = denom.to_numpy() if hasattr(denom, "to_numpy") else denom
        num_arr = num.to_numpy() if hasattr(num, "to_numpy") else num
        out = np.where(denom_arr == 0, np.nan, num_arr / denom_arr)
        return pd.Series(out, index=getattr(num, "index", None)) if hasattr(num, "index") else out

    df = df.copy()
    df["resp_per_capita"] = safe_divide(df["pesticide_respiratory_kg"], df["population"])
    df["resp_per_cropland_acre"] = safe_divide(df["pesticide_respiratory_kg"], df["cropland_acres"])

    for kg_col in AGGREGATE_KG_PROXIES:
        if kg_col not in df.columns or kg_col == "pesticide_respiratory_kg":
            continue
        slug = kg_col.replace("pesticide_", "").replace("_kg", "")
        pc, pa = f"{slug}_per_capita", f"{slug}_per_cropland_acre"
        df[pc] = safe_divide(df[kg_col], df["population"])
        df[pa] = safe_divide(df[kg_col], df["cropland_acres"])
        df[f"log_{pc}"] = np.log1p(df[pc])
        df[f"log_{pa}"] = np.log1p(df[pa])

    df["op_per_capita"] = safe_divide(df["pesticide_organophosphate_kg"], df["population"])
    df["op_per_cropland_acre"] = safe_divide(df["pesticide_organophosphate_kg"], df["cropland_acres"])
    df["carbamate_per_capita"] = safe_divide(df["pesticide_carbamate_kg"], df["population"])
    df["carbamate_per_cropland_acre"] = safe_divide(df["pesticide_carbamate_kg"], df["cropland_acres"])
    df["pyrethroid_per_capita"] = safe_divide(df["pesticide_pyrethroid_kg"], df["population"])
    df["pyrethroid_per_cropland_acre"] = safe_divide(df["pesticide_pyrethroid_kg"], df["cropland_acres"])

    for col in [
        "resp_per_capita", "resp_per_cropland_acre", "op_per_capita", "op_per_cropland_acre",
        "carbamate_per_capita", "carbamate_per_cropland_acre",
        "pyrethroid_per_capita", "pyrethroid_per_cropland_acre",
    ]:
        df[f"log_{col}"] = np.log1p(df[col])

    df["rural_binary"] = (df["nchs_urban_rural"] >= 5).astype(int)
    return df


AGGREGATE_KG_PROXIES = [
    "pesticide_respiratory_kg", "pesticide_total_kg", "pesticide_chlorophenoxy_kg",
    "pesticide_triazine_kg", "pesticide_anilide_kg", "pesticide_organochlorine_kg", "pesticide_other_kg",
]


def _aggregate_log_intensity_feature_names():
    names = ["log_resp_per_capita", "log_resp_per_cropland_acre"]
    for kg_col in AGGREGATE_KG_PROXIES:
        if kg_col == "pesticide_respiratory_kg":
            continue
        slug = kg_col.replace("pesticide_", "").replace("_kg", "")
        names.extend([f"log_{slug}_per_capita", f"log_{slug}_per_cropland_acre"])
    return names


AGGREGATE_LOG_INTENSITY = _aggregate_log_intensity_feature_names()
PEST_AGGREGATE_ORDER = list(dict.fromkeys(AGGREGATE_KG_PROXIES + AGGREGATE_LOG_INTENSITY))
PEST_COMPONENTS = [
    "log_op_per_capita", "log_op_per_cropland_acre",
    "log_carbamate_per_capita", "log_carbamate_per_cropland_acre",
    "log_pyrethroid_per_capita", "log_pyrethroid_per_cropland_acre",
]
COMPONENT_KG_RAW = [
    "pesticide_organophosphate_kg", "pesticide_carbamate_kg", "pesticide_pyrethroid_kg",
]

DEMO_COLS = [
    "population", "median_age", "median_income", "pct_white", "pct_black",
    "pct_asian", "pct_hispanic", "rural_binary",
]
HEALTH_CONFOUNDER_COLS = ["CSMOKING", "OBESITY", "DIABETES"]
CROPLAND_COLS = ["cropland_diversity", "county_crop_concentration", "pct_cropland"]
OTHER_COLS = ["YEAR"]
BASE_COLS = DEMO_COLS + HEALTH_CONFOUNDER_COLS + CROPLAND_COLS + OTHER_COLS

EXPOSURE_SETS = {
    "Aggs_raw": list(dict.fromkeys(AGGREGATE_KG_PROXIES + BASE_COLS)),
    "Aggs_engineered": list(dict.fromkeys(AGGREGATE_KG_PROXIES + AGGREGATE_LOG_INTENSITY + BASE_COLS)),
    "Components_raw": list(dict.fromkeys(COMPONENT_KG_RAW + BASE_COLS)),
    "Components_engineered": list(dict.fromkeys(PEST_COMPONENTS + BASE_COLS)),
}
# Full_pesticides_raw is built per-split in the data cell (all pesticide_*_kg + BASE_COLS).
FULL_PESTICIDES_RAW_KEY = "Full_pesticides_raw"
EXPOSURE_SET_KEYS = list(EXPOSURE_SETS.keys()) + [FULL_PESTICIDES_RAW_KEY]


# --- Full_pesticides_raw only: kitchen sink ---

FULL_KEY = "Full_pesticides_raw"
EXPOSURE_SET_KEYS = [FULL_KEY]

data = {}

# Full_pesticides_raw only: kitchen sink
FULL_KEY = "Full_pesticides_raw"

# CASTHMA
TARGET = "CASTHMA"
train_df = pd.read_csv(TARGET_PATHS[TARGET]["train"])
test_df = pd.read_csv(TARGET_PATHS[TARGET]["test"])
etr = engineer_signal_isolation_features(train_df.copy())
ete = engineer_signal_isolation_features(test_df.copy())

pest_kg_cols = sorted(
    c for c in train_df.columns
    if str(c).startswith("pesticide_") and str(c).endswith("_kg")
)

full_raw_cols = [
    c for c in list(dict.fromkeys(pest_kg_cols + BASE_COLS))
    if c in etr.columns and c in ete.columns
]

X_tr = etr[full_raw_cols].copy()
X_te = ete[full_raw_cols].copy()

data[TARGET] = {
    "train_df": train_df,
    "test_df": test_df,
    "train_eng": etr,
    "test_eng": ete,
    "X_by_exposure": {FULL_KEY: (X_tr, X_te)},
    "X_train": X_tr,
    "y_train": train_df[TARGET].copy(),
    "y_test": test_df[TARGET].copy(),
    "groups": train_df["FIPS"].copy(),
    "feature_columns": full_raw_cols,
}

print(f"{TARGET}: train {len(train_df)}, test {len(test_df)}, Full_pesticides_raw cols {X_tr.shape[1]}")

# COPD
TARGET = "COPD"
train_df = pd.read_csv(TARGET_PATHS[TARGET]["train"])
test_df = pd.read_csv(TARGET_PATHS[TARGET]["test"])
etr = engineer_signal_isolation_features(train_df.copy())
ete = engineer_signal_isolation_features(test_df.copy())

pest_kg_cols = sorted(
    c for c in train_df.columns
    if str(c).startswith("pesticide_") and str(c).endswith("_kg")
)

full_raw_cols = [
    c for c in list(dict.fromkeys(pest_kg_cols + BASE_COLS))
    if c in etr.columns and c in ete.columns
]

X_tr = etr[full_raw_cols].copy()
X_te = ete[full_raw_cols].copy()

data[TARGET] = {
    "train_df": train_df,
    "test_df": test_df,
    "train_eng": etr,
    "test_eng": ete,
    "X_by_exposure": {FULL_KEY: (X_tr, X_te)},
    "X_train": X_tr,
    "y_train": train_df[TARGET].copy(),
    "y_test": test_df[TARGET].copy(),
    "groups": train_df["FIPS"].copy(),
    "feature_columns": full_raw_cols,
}

print(f"{TARGET}: train {len(train_df)}, test {len(test_df)}, Full_pesticides_raw cols {X_tr.shape[1]}")


---

## Model-by-model outputs (Full_pesticides_raw)

Each cell fits one model family in increasing complexity order; you should see per-target RMSE/MAE/R2 as the cell runs.


In [ ]:
# Helper utilities + result collection for model-by-model outputs

def make_strata(y: pd.Series, n_bins: int = 5) -> pd.Series:
    n_unique = y.nunique(dropna=True)
    bins = max(2, min(n_bins, n_unique))
    return pd.qcut(y, q=bins, labels=False, duplicates="drop")


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )


SI_XGB_PARAM_GRID = {
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__n_estimators": [100, 200],
}

rows = []


## [1/7] Simple Linear Regression

Single-feature linear model (uses `pesticide_total_kg` if present).


In [ ]:
print("=== [1/7] Simple LR (1 feat) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

feature_cols = list(X_tr.columns)
pest_kg_cols = sorted([c for c in feature_cols if str(c).startswith("pesticide_") and str(c).endswith("_kg")])
single_pest = "pesticide_total_kg" if "pesticide_total_kg" in feature_cols else pest_kg_cols[0]

imp_single = SimpleImputer(strategy="median")
x_tr_s = imp_single.fit_transform(X_tr[[single_pest]])
x_te_s = imp_single.transform(X_te[[single_pest]])

m = LinearRegression().fit(x_tr_s, y_tr)
pred = m.predict(x_te_s)

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 1,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "Simple LR (1 feat)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

feature_cols = list(X_tr.columns)
pest_kg_cols = sorted([c for c in feature_cols if str(c).startswith("pesticide_") and str(c).endswith("_kg")])
single_pest = "pesticide_total_kg" if "pesticide_total_kg" in feature_cols else pest_kg_cols[0]

imp_single = SimpleImputer(strategy="median")
x_tr_s = imp_single.fit_transform(X_tr[[single_pest]])
x_te_s = imp_single.transform(X_te[[single_pest]])

m = LinearRegression().fit(x_tr_s, y_tr)
pred = m.predict(x_te_s)

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 1,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "Simple LR (1 feat)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})


## [2/7] Multiple Linear Regression (Ridge, numeric features)

Ridge on numeric columns only.


In [ ]:
print("=== [2/7] Multiple LR (Ridge, numeric) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

numeric_cols = X_tr.select_dtypes(include=["number", "bool"]).columns.tolist()
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])

pipe.fit(X_tr[numeric_cols], y_tr)
pred = pipe.predict(X_te[numeric_cols])

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 2,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "Multiple LR (Ridge, numeric)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

numeric_cols = X_tr.select_dtypes(include=["number", "bool"]).columns.tolist()
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])

pipe.fit(X_tr[numeric_cols], y_tr)
pred = pipe.predict(X_te[numeric_cols])

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 2,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "Multiple LR (Ridge, numeric)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})


## [3/7] Multiple Linear Regression (Ridge, full preprocess)

Ridge on the full preprocessing pipeline (numeric + one-hot for categoricals).


In [ ]:
print("=== [3/7] Multiple LR (Ridge, full preprocess) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

preproc = build_preprocessor(X_tr)
X_tr_f = preproc.fit_transform(X_tr)
X_te_f = preproc.transform(X_te)

m = Ridge(alpha=1.0).fit(X_tr_f, y_tr)
pred = m.predict(X_te_f)

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 3,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "Multiple LR (Ridge, full preprocess)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

preproc = build_preprocessor(X_tr)
X_tr_f = preproc.fit_transform(X_tr)
X_te_f = preproc.transform(X_te)

m = Ridge(alpha=1.0).fit(X_tr_f, y_tr)
pred = m.predict(X_te_f)

rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")
rows.append({
    "complexity": 3,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "Multiple LR (Ridge, full preprocess)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})


## [4/7] Ridge (tuned)

GridSearchCV over `alpha` with the same grouped CV strategy as the main notebook.


In [ ]:
print("=== [4/7] Ridge (tuned) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", Ridge())]),
    param_grid={"model__alpha": ALPHA_GRID},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha}")
rows.append({
    "complexity": 4,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "Ridge (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", Ridge())]),
    param_grid={"model__alpha": ALPHA_GRID},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha}")
rows.append({
    "complexity": 4,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "Ridge (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})


## [5/7] Lasso (tuned)

GridSearchCV over `alpha` with grouped CV.


In [ ]:
print("=== [5/7] Lasso (tuned) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", Lasso(max_iter=20000, random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": ALPHA_GRID},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha}")
rows.append({
    "complexity": 5,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "Lasso (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", Lasso(max_iter=20000, random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": ALPHA_GRID},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha}")
rows.append({
    "complexity": 5,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "Lasso (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": np.nan,
    "best_params": np.nan,
})


## [6/7] ElasticNet (tuned)

GridSearchCV over `alpha` and `l1_ratio` with grouped CV.


In [ ]:
print("=== [6/7] ElasticNet (tuned) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", ElasticNet(max_iter=20000, random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": ALPHA_GRID, "model__l1_ratio": [0.25, 0.5, 0.75]},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
best_l1 = search.best_params_["model__l1_ratio"]
print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha} best_l1_ratio={best_l1}")
rows.append({
    "complexity": 6,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "ElasticNet (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": best_l1,
    "best_params": np.nan,
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", ElasticNet(max_iter=20000, random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": ALPHA_GRID, "model__l1_ratio": [0.25, 0.5, 0.75]},
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_alpha = search.best_params_["model__alpha"]
best_l1 = search.best_params_["model__l1_ratio"]
print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_alpha={best_alpha} best_l1_ratio={best_l1}")
rows.append({
    "complexity": 6,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "ElasticNet (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": best_alpha,
    "best_l1_ratio": best_l1,
    "best_params": np.nan,
})


## [7/7] XGBoost (tuned)

GridSearchCV over depth/learning_rate/n_estimators.


In [ ]:
print("=== [7/7] XGBoost (tuned) ===")

# CASTHMA
print("-- CASTHMA --")
d = data["CASTHMA"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
    param_grid=SI_XGB_PARAM_GRID,
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_params = search.best_params_
print(f"CASTHMA: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_params={best_params}")
rows.append({
    "complexity": 7,
    "target": "CASTHMA",
    "exposure_set": FULL_KEY,
    "model": "XGBoost (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": repr(best_params),
})

# COPD
print("-- COPD --")
d = data["COPD"]
y_tr, y_te = d["y_train"], d["y_test"]
X_tr, X_te = d["X_by_exposure"][FULL_KEY]

strata = make_strata(y_tr)
folds = list(
    StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    .split(d["X_train"], strata, d["groups"])
)

preproc = build_preprocessor(X_tr)
search = GridSearchCV(
    Pipeline([("preprocess", preproc), ("model", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
    param_grid=SI_XGB_PARAM_GRID,
    cv=folds,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
).fit(X_tr, y_tr)

pred = search.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))

best_params = search.best_params_
print(f"COPD: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f} best_params={best_params}")
rows.append({
    "complexity": 7,
    "target": "COPD",
    "exposure_set": FULL_KEY,
    "model": "XGBoost (tuned)",
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_alpha": np.nan,
    "best_l1_ratio": np.nan,
    "best_params": repr(best_params),
})


In [ ]:
print("=== Summary (sorted by complexity) ===")
exposure_summary_all = pd.DataFrame(rows)
display(exposure_summary_all.sort_values(["complexity", "target"]).reset_index(drop=True))
